# Lecture: Stable Diffusion — Putting It All Together

Throughout Chapter C4 we built every component of a modern text-to-image model
**from scratch** on Fashion-MNIST:

- **C4-2** — a U-Net that predicts noise (the DDPM training objective).
- **C4-3** — fast **DDIM** sampling (generate in ~50 steps instead of 1000).
- **C4-4** — **conditioning + classifier-free guidance** (the `guidance_scale`).
- **C4-5** — **latent diffusion**: diffuse in a compressed autoencoder space.

This final notebook shows the **real thing**: **Stable Diffusion 1.5**, a model
trained on hundreds of millions of image–text pairs. The key message is *not*
"here is a magic black box" — it is:

> **You already understand every part of it.** Stable Diffusion is exactly the
> architecture you built, only larger and conditioned on **text** instead of class
> labels.

We will load the pre-trained pipeline (**no training**, like the StyleGAN notebook
C3-2), generate images from text prompts, **open up the pipeline** to identify each
component you know, and reproduce the `guidance_scale` and sampling-step
experiments from C4-3/C4-4 — now on photorealistic images.

> **Requirements:** This notebook needs a **GPU** (Colab: Runtime → Change runtime
> type → T4 GPU). The first cell downloads ~4 GB of model weights, so it takes a
> few minutes. Everything afterwards is fast.

### Install pinned dependencies

Stable Diffusion lives in the `diffusers` library. We pin a set of mutually
compatible versions so the notebook keeps working as the libraries evolve (their
APIs change often).

> **If Colab prompts you to restart the runtime after this cell, do so** (Runtime
> → Restart session), then continue from the next cell — the upgraded
> `transformers` / `huggingface_hub` only take effect after a restart.

In [ ]:
!pip install -q diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2

### Load the Stable Diffusion 1.5 pipeline

`StableDiffusionPipeline` bundles all four components (autoencoder, U-Net, text
encoder, scheduler) behind a single call. We load it in half precision (`fp16`) to
fit comfortably on a T4 GPU.

The safety checker is disabled purely to avoid false positives on harmless prompts
in a teaching setting.

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline

assert torch.cuda.is_available(), "This notebook requires a GPU runtime (Colab: T4)."
device = "cuda"

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)

print("Stable Diffusion 1.5 loaded on", device)

### A first image

The canonical Stable Diffusion demo prompt. We fix the random seed with a
`torch.Generator` so every run produces the **same** image — exactly the
deterministic-sampling idea from C4-3 (a fixed starting noise gives a reproducible
result).

In [ ]:
prompt = ("a photograph of an astronaut riding a horse, "
          "highly detailed, sharp focus, cinematic lighting")

generator = torch.Generator(device=device).manual_seed(42)
image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5,
             generator=generator).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("astronaut riding a horse", fontsize=10)
plt.show()

## Opening up the pipeline

The "magic" pipeline is just the four components you built in this chapter. Let us
print them and their sizes — and map each one back to your own notebooks:

| Pipeline component | What it is | You built it in |
|---|---|---|
| `pipe.vae` | Autoencoder: image ↔ latent | **C4-5** (`AutoEncoder`) |
| `pipe.unet` | Noise-prediction U-Net (in latent space) | **C4-2 / C4-5** |
| `pipe.text_encoder` | CLIP: text prompt → conditioning embedding | **new** (replaces the class embedding of **C4-4**) |
| `pipe.scheduler` | Sampling rule (DDIM-like) | **C4-3** |

The *only* genuinely new piece is the **text encoder**: where C4-4 used an
`nn.Embedding` over 10 class labels, Stable Diffusion uses a **CLIP text encoder**
that turns an arbitrary prompt into a conditioning vector. Everything else is the
machinery you already implemented.

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print(f"VAE (autoencoder):   {count_params(pipe.vae):>12,} params   <- C4-5")
print(f"U-Net (denoiser):    {count_params(pipe.unet):>12,} params   <- C4-2 / C4-5")
print(f"Text encoder (CLIP): {count_params(pipe.text_encoder):>12,} params   <- new (replaces C4-4 class embedding)")
print(f"Scheduler:           {type(pipe.scheduler).__name__:>12}   <- C4-3 (DDIM-family)")

### The latent space is real

Stable Diffusion diffuses in a `4×64×64` latent (for 512×512 images) — a 48×
spatial compression, exactly the latent-diffusion idea from **C4-5**, just bigger
than our `4×7×7`. We can confirm this by inspecting the VAE's latent channels and
the scaling factor it uses.

In [ ]:
print("VAE latent channels:", pipe.vae.config.latent_channels)
print("VAE downsampling factor:", pipe.vae_scale_factor, "(512 / 8 = 64 -> latent is 4x64x64)")
print("Scheduler:", pipe.scheduler.compatibles[0].__name__, "and relatives (DDIM-family)")

## Experiment 1 — The guidance scale (CFG)

This is the **same** classifier-free guidance you implemented in C4-4 and C4-5,
now steering a photorealistic image. We fix the prompt and seed and sweep
`guidance_scale`:

- low $w$ (≈ 1): the model barely follows the prompt — generic, washed-out.
- medium $w$ (≈ 7.5, the SD default): faithful to the prompt, good quality.
- high $w$ (≈ 20): over-saturated, distorted — the same over-guiding artefacts you
  saw on Fashion-MNIST.

The effect is identical to C4-4; only the images are real photos now.

In [ ]:
prompt = ("a red sports car on a mountain road at sunset, "
          "highly detailed, sharp focus, professional photograph")
scales = [1.0, 3.0, 7.5, 20.0]

fig, axes = plt.subplots(1, len(scales), figsize=(18, 5))
for ax, w in zip(axes, scales):
    generator = torch.Generator(device=device).manual_seed(0)  # same seed each time
    img = pipe(prompt, num_inference_steps=50, guidance_scale=w,
               generator=generator).images[0]
    ax.imshow(img)
    ax.set_title(f"guidance = {w}", fontsize=11)
    ax.axis("off")
plt.suptitle(f'"{prompt}"', y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

## Experiment 2 — Number of sampling steps (DDIM)

`num_inference_steps` is precisely the DDIM step count from **C4-3**. Fewer steps
means faster generation but coarser detail. SD's default scheduler is a DDIM
relative, so the speed/quality trade-off you measured on Fashion-MNIST applies
here too — watch the feather detail degrade as steps drop.

In [ ]:
prompt = ("a highly detailed portrait of an owl, sharp feathers, "
          "studio lighting, intricate detail, 8k, sharp focus")
step_counts = [50, 20, 10]

fig, axes = plt.subplots(1, len(step_counts), figsize=(15, 5))
for ax, steps in zip(axes, step_counts):
    generator = torch.Generator(device=device).manual_seed(7)  # same seed each time
    img = pipe(prompt, num_inference_steps=steps, guidance_scale=7.5,
               generator=generator).images[0]
    ax.imshow(img)
    ax.set_title(f"{steps} steps", fontsize=11)
    ax.axis("off")
plt.suptitle(f'"{prompt}"', y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

## Experiment 3 — Negative prompts

A `negative_prompt` lists what you do **not** want to see. Technically it replaces
the *unconditional* branch of classifier-free guidance (C4-4): instead of guiding
away from an empty prompt, the model guides away from the negative prompt. This is
the single most effective quality lever for SD 1.5 — it suppresses common failure
modes like blur, distortion and bad anatomy.

We generate the same prompt and seed **without** and **with** a negative prompt to
see the difference directly.

In [ ]:
prompt = ("a portrait photograph of a golden retriever puppy, "
          "highly detailed, sharp focus, natural light")
negative = "blurry, low quality, distorted, deformed, ugly, low resolution"

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, neg, label in zip(axes, ["", negative], ["no negative prompt", "with negative prompt"]):
    generator = torch.Generator(device=device).manual_seed(3)  # same seed
    img = pipe(prompt, negative_prompt=neg, num_inference_steps=50,
               guidance_scale=7.5, generator=generator).images[0]
    ax.imshow(img)
    ax.set_title(label, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Experiment 4 — A better sampler

The scheduler (C4-3) decides *how* the reverse process is discretised. SD 1.5's
default is a decent but basic sampler; modern solvers like
**`DPMSolverMultistepScheduler`** reach comparable quality in **fewer** steps, or
better quality at the same step count. Swapping it is a one-liner — the trained
U-Net is untouched, exactly the "training and sampling are decoupled" idea from
C4-3.

We render the same prompt and seed at **20 steps** with the default scheduler and
with DPM-Solver, so the quality difference at a low step budget is visible.

In [ ]:
from diffusers import DPMSolverMultistepScheduler

prompt = ("a highly detailed portrait of an owl, sharp feathers, "
          "studio lighting, intricate detail, 8k, sharp focus")

# Keep a reference to the original scheduler so we can restore it.
default_scheduler = pipe.scheduler

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# (1) Default scheduler at 20 steps
pipe.scheduler = default_scheduler
generator = torch.Generator(device=device).manual_seed(7)
img_default = pipe(prompt, num_inference_steps=20, guidance_scale=7.5,
                   generator=generator).images[0]
axes[0].imshow(img_default); axes[0].set_title("default scheduler, 20 steps", fontsize=11)
axes[0].axis("off")

# (2) DPM-Solver at 20 steps
pipe.scheduler = DPMSolverMultistepScheduler.from_config(default_scheduler.config)
generator = torch.Generator(device=device).manual_seed(7)
img_dpm = pipe(prompt, num_inference_steps=20, guidance_scale=7.5,
               generator=generator).images[0]
axes[1].imshow(img_dpm); axes[1].set_title("DPM-Solver, 20 steps", fontsize=11)
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Restore the default scheduler for the cells that follow.
pipe.scheduler = default_scheduler

## Your turn — generate from your own prompt

Edit the `my_prompt` string below and run the cell. Try changing the
`guidance_scale` (5–10 usually works well) and the seed to explore different
images for the same prompt.

Tips for good prompts: be specific, add style/lighting/quality cues
("oil painting", "golden hour", "cinematic", "highly detailed", "sharp focus"),
and name the subject clearly. Use `my_negative` to suppress unwanted traits
(blur, distortion, bad anatomy).

In [ ]:
my_prompt = "a cozy wooden cabin in a snowy forest, warm light in the windows, digital art"
my_negative = "blurry, low quality, distorted, deformed"

seed = 1234
guidance = 7.5
steps = 50

generator = torch.Generator(device=device).manual_seed(seed)
image = pipe(my_prompt, negative_prompt=my_negative, num_inference_steps=steps,
             guidance_scale=guidance, generator=generator).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title(my_prompt, fontsize=9)
plt.show()

## Summary — the whole chapter in one model

Stable Diffusion is the sum of everything in Chapter C4:

| Ingredient | Notebook | Role in Stable Diffusion |
|---|---|---|
| Noise-prediction U-Net + DDPM loss | C4-2 | The core denoiser |
| DDIM fast sampling | C4-3 | `num_inference_steps` |
| Conditioning + classifier-free guidance | C4-4 | `guidance_scale`, prompt control |
| Latent diffusion (autoencoder) | C4-5 | Diffusing `4×64×64` latents, not pixels |
| **Text encoder (CLIP)** | C4-6 | Turns prompts into conditioning |

The leap from your Fashion-MNIST models to Stable Diffusion is **scale and data**,
not new concepts. The single new architectural idea here — conditioning on text via
a CLIP encoder — is a drop-in replacement for the class embedding of C4-4. You now
understand, end to end, how a modern text-to-image model works.

---
## "But this looks worse than ChatGPT or Gemini!" — Why?

If you have used **GPT-4o image generation**, **Gemini "Nano Banana"** or
**Midjourney**, the images above probably look dated: less sharp, weaker at
following the prompt, prone to mangled hands and text. That gap is real — and it
is worth understanding, because it tells you what actually drives image quality.
**It is not new concepts.** Every component you see here is still inside those
modern models; they have just been scaled up and improved along five axes:

1. **Age and size.** SD 1.5 is from **2022** with an ~860M-parameter U-Net.
   The 2025 models are one to two orders of magnitude larger and trained on far
   more compute — comparable to the jump from an early small language model to a
   frontier LLM.

2. **The text encoder is the biggest single factor.** SD 1.5 reads your prompt
   with **CLIP** (a 77-token contrastive encoder with no real language
   understanding) — it essentially keyword-matches. That is *why* this notebook
   needs crutches like `"highly detailed, sharp focus"` and negative prompts.
   Modern systems condition on **large language models**, so they actually parse
   composition, counting, spatial relations and text-in-image. Better prompt
   *understanding*, not just better pixels.

3. **A different architecture class.** SD 1.5 is a **U-Net** latent diffusion
   model — exactly what you built. The current state of the art replaces the
   U-Net with a **Diffusion Transformer (DiT)**, which scales much better
   (SD3, FLUX). And GPT-4o / Nano Banana are not even standalone diffusion
   pipelines: image generation is **fused into a multimodal LLM**, so the same
   model that understands the text also produces the image — hence their strong
   prompt-following and consistent editing.

4. **Data and preference tuning.** Modern models are trained on **re-captioned**
   datasets (images relabeled by vision-LLMs) and then **tuned on human
   preference** (RLHF / aesthetic fine-tuning). That last step alone explains
   much of the "looks polished" difference.

5. **Resolution.** SD 1.5 generates natively at **512×512**; today's models work
   natively at 1024² and higher with stronger autoencoders.

**The takeaway:** the leap to ChatGPT- or Gemini-quality images is **scale + an
LLM text encoder + a new architecture (DiT or multimodal) + preference tuning** —
*not* a different idea about how diffusion works. Everything you built in C4 is
still the foundation; we used SD 1.5 here precisely because it is the one model
recent enough to be useful yet still a clean four-component pipeline you can open
up and recognise part by part. You could not dissect Nano Banana this way — there
is no separate U-Net, scheduler and text encoder to point at.

---
## Try It Yourself — Explore Stable Diffusion

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Prompt engineering.** Take your own prompt and add style cues one at a time
("oil painting", "cinematic lighting", "highly detailed"). Which cue changes the
image most? What does this tell you about how much the *text encoder* (not the
U-Net) drives the result?

**B. Find the guidance sweet spot.** For a prompt of your choice, sweep
`guidance_scale` over 1, 5, 7.5, 12, 20. At which value is the image both faithful
*and* natural-looking? Relate the over-guided result directly to the over-guiding
you saw on Fashion-MNIST in C4-4.

**C. Steps vs. quality.** Lower `num_inference_steps` to 5. Is the image still
coherent? Compare this "breaking point" to the one you found for the from-scratch
DDIM model in C4-3 — why might the real model tolerate fewer steps?

**D. Same seed, same image.** Fix the seed and run the *same* prompt twice — the
images are identical. Now change only the seed. Explain, using C4-3's
deterministic-sampling idea, why the seed fully determines the image while the
prompt is held constant.

**E. Locate the components.** Without looking at the table above, write down which
`pipe.*` attribute corresponds to (i) the autoencoder from C4-5, (ii) the denoiser
from C4-2, (iii) the guidance mechanism from C4-4. Verify with the "Opening up the
pipeline" cell.

**F. Negative prompts & samplers.** Add aggressive terms to `my_negative`
(e.g. "cartoon, painting, text, watermark") and observe the effect. Then, using
Experiment 4, switch to DPM-Solver and drop the steps to 15 — do you still get a
clean image? Which lever (negative prompt, sampler, steps) gave you the biggest
quality gain for the least effort?